# 05 — Model Explainability with SHAP

**SHAP (SHapley Additive exPlanations)** is a game-theoretic framework for explaining the output of any machine learning model. It answers the question: *"How much did each feature contribute to this particular prediction?"*

Accuracy metrics like ROC AUC tell us *how good* a model is — but they don't tell us *why* it makes the decisions it makes. SHAP fills that gap. It assigns each feature a value (a 'SHAP value') for each individual prediction, representing the feature's marginal contribution after accounting for all possible feature orderings. Positive SHAP values push the prediction toward churn; negative values push toward retention.

**Why this matters in practice:** A product team can't act on 'XGBoost says this customer will churn with 74% probability.' They *can* act on 'This customer's month-to-month contract and three support tickets in the last 30 days are the two biggest drivers of their high churn risk.' SHAP translates model outputs into human-readable causal narratives.

## 1. Setup — Load Model & Data

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../')

from src.preprocessing import get_model_features
from src.model import split, save_model

# Load the saved model
import pickle
with open('../models/xgboost_churn.pkl', 'rb') as f:
    model = pickle.load(f)

print(f'Model loaded: {type(model).__name__}')

# Load processed data and prepare features
df = pd.read_csv('../data/processed/customers_processed.csv')
X, y = get_model_features(df)
X_train, X_test, y_train, y_test = split(X, y)

print(f'Test set: {X_test.shape[0]:,} customers, {X_test.shape[1]} features')

## 2. Compute SHAP Values

`shap.TreeExplainer` is optimised for tree-based models like XGBoost and is dramatically faster than the model-agnostic `KernelExplainer`. It computes exact SHAP values (not approximations) by exploiting the tree structure.

The resulting `shap_values` array has the same shape as `X_test` — one SHAP value per customer per feature.

In [ ]:
import shap

# Initialise the TreeExplainer with our XGBoost model
explainer = shap.TreeExplainer(model)

print('Computing SHAP values for test set...')
shap_values = explainer.shap_values(X_test)

print(f'SHAP values shape: {shap_values.shape}')
print(f'Expected value (base rate): {explainer.expected_value:.4f}')
print(f'  (This is the model\'s average log-odds prediction before considering any features)')

## 3. Global Feature Importance (Bar Plot)

The bar summary plot shows **mean absolute SHAP value** for each feature — i.e., on average, how much does each feature shift the model's output? This is a more theoretically grounded importance measure than XGBoost's built-in importance, because it's denominated in the same units as the model's output (log-odds of churn).

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_test,
    plot_type='bar',
    show=False
)
plt.title('Global Feature Importance (Mean |SHAP Value|)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

## 4. Beeswarm Plot — Direction of Effects

The beeswarm plot overlays every individual SHAP value for every customer in the test set. Each dot is one customer. The colour represents the feature value (red = high, blue = low), and the horizontal position shows the SHAP contribution (right = pushes toward churn, left = pushes toward retention).

This plot is far richer than a simple importance bar — it shows **both magnitude and direction** simultaneously. For instance, a feature where all red dots are on the right tells you that high values of that feature strongly predict churn.

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_test,
    show=False
)
plt.title('SHAP Beeswarm Plot — Feature Effects on Churn Prediction', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

## 5. Single Customer Explanation — Force Plot

While global plots show average behaviour, the force plot explains **one individual prediction**. It visualises the tug-of-war between features pushing the prediction up (toward churn) and features pulling it down (toward retention).

We select the customer in the test set with the highest predicted churn probability — this is the customer a retention team would prioritise calling first.

In [ ]:
# Find the highest-risk customer in the test set
y_proba_test = model.predict_proba(X_test)[:, 1]
highest_risk_idx = np.argmax(y_proba_test)

print(f'Highest-risk customer in test set:')
print(f'  Index: {highest_risk_idx}')
print(f'  Predicted churn probability: {y_proba_test[highest_risk_idx]:.1%}')
print(f'  Actual outcome: {"Churned" if y_test.iloc[highest_risk_idx] == 1 else "Retained"}')
print()
print('Feature values for this customer:')
print(X_test.iloc[highest_risk_idx].to_string())

In [ ]:
# Force plot for the highest-risk customer
plt.figure(figsize=(14, 4))
shap.force_plot(
    explainer.expected_value,
    shap_values[highest_risk_idx],
    X_test.iloc[highest_risk_idx],
    matplotlib=True,
    show=False
)
plt.title(f'Force Plot — Customer at Index {highest_risk_idx} (Predicted Churn: {y_proba_test[highest_risk_idx]:.1%})', 
          fontsize=11, pad=20)
plt.tight_layout()
plt.show()

## 6. Top SHAP Findings in Business Language

Raw SHAP values are meaningless to a stakeholder who doesn't know what a log-odds shift of 0.3 means. Here we translate the top 5 findings from the beeswarm plot into plain English:

---

**1. Contract Type (Month-to-Month) — The #1 churn driver**
Customers on month-to-month contracts have the highest positive SHAP values of any feature — this single attribute pushes the predicted churn probability higher than any other variable in the model. They face zero friction to cancelling, and the model has learned that this structural flexibility translates directly into elevated churn risk.

**2. Tenure — Strong protective effect, but only after ~12 months**
Low-tenure customers (shown as blue dots on the right side of the beeswarm) have high SHAP values for this feature — short tenure strongly predicts churn. Conversely, high-tenure customers (red dots, left side) are substantially protected. The 12-month mark appears to be the approximate inflection point where customers transition from 'at risk' to 'loyal'.

**3. Monthly Charges — Non-linear relationship**
The beeswarm shows that very high and very low monthly charges both have notable SHAP effects, but in complex ways. Customers paying significantly above average (perhaps on premium plans they don't fully utilise) show elevated churn risk — a signal that perceived value-for-money erosion is happening at the high end.

**4. Tech Support & Online Security Add-ons — Significant protective factors**
Customers who have purchased tech support or online security add-ons show negative SHAP values (pushes toward retention). This may reflect both product stickiness (more integrated into the product) and higher engagement. Upselling these services to at-risk customers could simultaneously increase ARPU and reduce churn.

**5. Paperless Billing — Correlates with higher churn**
Counterintuitively, paperless billing — often associated with more digitally engaged customers — correlates positively with churn. This may reflect that customers who opted into digital-first interaction are also more comfortable with digital-first cancellations, or that this variable is a proxy for a particular customer acquisition channel.

## 7. Actionable Recommendations

Based on the SHAP analysis, here are concrete product and GTM recommendations ranked by expected impact:

---

**Priority 1: Aggressive contract upgrade incentives for month-to-month customers**
Since contract type is the dominant churn driver, the highest-ROI intervention is converting month-to-month customers to annual contracts. A discount equivalent to even 1–2 months of MRR more than pays for itself if it prevents a churn. Identify the month-to-month customers with the top 20% churn probability scores and offer them a targeted annual contract promotion.

**Priority 2: 90-day onboarding programme for new customers**
The tenure SHAP curve makes clear that survival through the first 90 days is the most critical predictor of long-term retention. A structured onboarding programme with proactive check-ins, usage milestone celebrations, and a dedicated CSM touch at day 30 and day 60 would directly address the front-loaded churn pattern identified in the cohort analysis.

**Priority 3: Upsell tech support & security features to high-risk segments**
The protective SHAP effect of add-on products creates a double-win opportunity: a targeted upsell campaign that reduces churn *and* increases ARPU simultaneously. Focus on customers who score high on the churn model but don't yet have these features.

**Priority 4: Value communication for high-spend customers**
High monthly charges driving elevated churn suggests perceived value erosion. A quarterly business review programme for customers in the top spending quartile — proactively showing ROI and feature utilisation — could materially reduce churn in the segment that matters most for MRR.